<a href="https://colab.research.google.com/github/TU-USUARIO/labo1-colabs/blob/main/09_Senales_periodicas_y_determinacion_del_periodo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 09 — Señales periódicas: detección de picos y determinación del período**Laboratorio 1 · Clase 9****Objetivos.**1. Detectar máximos en una señal adquirida con `scipy.signal.find_peaks` y entender sus parámetros.2. Descubrir por qué **promediar los intervalos entre picos desperdicia casi todos tus datos**.3. Determinar el período por ajuste lineal, que es el método correcto.**Requisitos previos:** Colabs 05 y 07.> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`. Vas a trabajar sobre *tu* copia; el original queda intacto para el resto del curso.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.signal import find_peaksfrom scipy.optimize import curve_fitnp.random.seed(20261007)

---## 1. La señal

In [ ]:
# --- DATOS DE EJEMPLO: oscilador masa-resorte (reemplazar por los propios) ---T_real = 0.784                       # sfs = 200.0                           # Hz de muestreot = np.arange(0, 12, 1/fs)sigma_x = 0.0008                     # mx = 0.042*np.cos(2*np.pi*t/T_real + 0.3) + np.random.normal(0, sigma_x, len(t))# -----------------------------------------------------------------------------fig, ax = plt.subplots(figsize=(9, 3.4))ax.plot(t, x*1000, lw=0.8)ax.set_xlabel('$t$ [s]'); ax.set_ylabel('$x$ [mm]')ax.set_title(f'Señal adquirida — {len(t)} puntos a {fs:.0f} Hz')ax.grid(alpha=0.3); fig.tight_layout(); plt.show()print(f"Puntos por período: {T_real*fs:.0f}  (criterio: al menos 10)")

---## 2. Detección de picos`find_peaks` devuelve los índices de los máximos locales. Sin argumentos encuentra **todos**,incluidos los que produce el ruido. Los tres parámetros que importan:- `height` — altura mínima absoluta. Simple, pero falla si la señal se amortigua.- `prominence` — cuánto sobresale el pico respecto de su entorno. **Es el más robusto** y el que  conviene usar por defecto.- `distance` — separación mínima en número de muestras. Útil si conocés aproximadamente el período:  `distance = 0.7 * T_esperado * fs`.

In [ ]:
print("sin argumentos:", len(find_peaks(x)[0]), "picos  <- incluye ruido")idx, props = find_peaks(x, prominence=0.5*x.max(), distance=int(0.7*T_real*fs))t_pico, x_pico = t[idx], x[idx]print(f"con prominence y distance: {len(idx)} picos")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))ax.plot(t, x*1000, lw=0.7, alpha=0.7)ax.plot(t_pico, x_pico*1000, 'v', color='crimson', ms=7, label='picos detectados')ax.set_xlabel('$t$ [s]'); ax.set_ylabel('$x$ [mm]')ax.grid(alpha=0.3); ax.legend(); fig.tight_layout(); plt.show()

> **Advertencia sobre la incerteza del pico.** El instante del máximo no se conoce mejor que> $1/f_s$: si muestreás a 200 Hz, cada tiempo de pico tiene una incerteza del orden de 5 ms. Con> señales muy ruidosas conviene ajustar una parábola a los tres puntos alrededor del máximo y tomar> su vértice — se gana casi un orden de magnitud.

---## 3. El método ingenuo, y por qué está malLo primero que a todos se nos ocurre: calcular los intervalos entre picos consecutivos ypromediarlos.

In [ ]:
intervalos = np.diff(t_pico)T_prom = intervalos.mean()dT_prom = np.std(intervalos, ddof=1)/np.sqrt(len(intervalos))print(f"cantidad de intervalos : {len(intervalos)}")print(f"T (promedio de Δt)     : {T_prom:.5f} ± {dT_prom:.5f} s")

Ahora mirá esto:

In [ ]:
n = len(t_pico)telescopico = (t_pico[-1] - t_pico[0]) / (n - 1)print(f"promedio de los intervalos            : {T_prom:.9f}")print(f"(t_último - t_primero) / (n_picos - 1): {telescopico:.9f}")print(f"¿son idénticos?  {np.isclose(T_prom, telescopico, rtol=1e-12)}")

**Son exactamente el mismo número.** No es casualidad: la suma de las diferencias consecutivas estelescópica,$$ \\sum_{i=1}^{n-1}(t_{i+1}-t_i) = t_n - t_1 $$así que el promedio de los intervalos usa **únicamente el primer y el último pico**. Los otrosdiecisiete se cancelan algebraicamente. Los medisite, los cargaste, y no intervienen en elresultado.Además, la incerteza que calculamos con `std/√n` está mal: los intervalos consecutivos comparten unextremo y por lo tanto **no son independientes**, con lo cual la fórmula del SEM no aplica.

---## 4. El método correcto: ajuste de $t_n$ vs. $n$Si el movimiento es periódico, el pico número $n$ ocurre en$$ t_n = T\\, n + t_0 $$Es una recta. Ajustarla por cuadrados mínimos usa **todos** los picos y devuelve $T$ como pendiente,con su incerteza calculada correctamente por el propio ajuste.

In [ ]:
def recta(n, T, t0):    return T*n + t0nums = np.arange(len(t_pico))sigma_pico = 1/fs                                   # incerteza del instante de cada picopopt, pcov = curve_fit(recta, nums, t_pico,                       sigma=np.full_like(t_pico, sigma_pico), absolute_sigma=True)T_aj, dT_aj = popt[0], np.sqrt(pcov[0, 0])print(f"T (ajuste de t_n vs n) : {T_aj:.5f} ± {dT_aj:.5f} s")print(f"T (promedio de Δt)     : {T_prom:.5f} ± {dT_prom:.5f} s")print(f"T verdadero            : {T_real:.5f} s")print(f"\nEl ajuste da una incerteza {dT_prom/dT_aj:.1f} veces menor.")

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(7, 5.2), sharex=True,                             gridspec_kw={'height_ratios': [2, 1]})a1.errorbar(nums, t_pico, yerr=sigma_pico, fmt='o', ms=5, capsize=3, label='picos')a1.plot(nums, recta(nums, *popt), 'crimson', lw=1.5, label=f'$t_n = T n + t_0$,  T = {T_aj:.4f} s')a1.set_ylabel('Instante del pico [s]'); a1.grid(alpha=0.3); a1.legend()a1.set_title('Determinación del período por ajuste')res = (t_pico - recta(nums, *popt))*1000a2.errorbar(nums, res, yerr=sigma_pico*1000, fmt='o', ms=5, capsize=3)a2.axhline(0, color='crimson', lw=1.2)a2.set_xlabel('Número de pico $n$'); a2.set_ylabel('residuo [ms]'); a2.grid(alpha=0.3)fig.subplots_adjust(hspace=0.08); plt.show()

El panel de residuos tiene además un uso específico acá: si el período **deriva** durante laadquisición (por ejemplo, porque la amplitud disminuye y el sistema no es perfectamente armónico),los residuos muestran curvatura. Es la manera más simple de detectar anarmonicidad.> **Ejercicio 9.1.** Sacá los picos intermedios (`nums[::3]`, `t_pico[::3]`) y volvé a ajustar. ¿Sube> mucho $\\sigma_T$? Compará con lo que le pasa al método del promedio de intervalos, que ya usaba> solo dos.

---## 5. Del período a la constante del resorteCon $T = 2\\pi\\sqrt{m_{ef}/k}$ y $m_{ef} = m + m_{resorte}/3$:$$ k = \\frac{4\\pi^2 m_{ef}}{T^2}, \\qquad   \\frac{\\sigma_k}{k} = \\sqrt{\\left(\\frac{\\sigma_{m_{ef}}}{m_{ef}}\\right)^2 + \\left(2\\frac{\\sigma_T}{T}\\right)^2} $$El factor 2 delante de $\\sigma_T/T$ viene de que $T$ está al cuadrado. Por eso conviene invertiresfuerzo en medir bien el período: cada mejora se duplica en el resultado.La corrección $m_{resorte}/3$ no es un detalle cosmético; está deducida en la literatura clásica deltema (ver los trabajos sobre la corrección por masa del resorte en *American Journal of Physics*) yen resortes livianos con masas chicas cambia $k$ por encima de la barra de error.

In [ ]:
m_col, dm_col = 0.2000, 0.0002          # kg, masa colgadam_res, dm_res = 0.0180, 0.0005          # kg, masa del resortem_ef = m_col + m_res/3dm_ef = np.sqrt(dm_col**2 + (dm_res/3)**2)k = 4*np.pi**2 * m_ef / T_aj**2dk = k*np.sqrt((dm_ef/m_ef)**2 + (2*dT_aj/T_aj)**2)print(f"m_ef = {m_ef:.5f} ± {dm_ef:.5f} kg")print(f"k    = {k:.2f} ± {dk:.2f} N/m")# sin la corrección:k_sin = 4*np.pi**2 * m_col / T_aj**2print(f"\nk sin corregir por la masa del resorte: {k_sin:.2f} N/m")print(f"diferencia: {abs(k-k_sin):.2f} N/m  =  {abs(k-k_sin)/dk:.1f} σ")

---## 6. Ejercicios**9.2.** Determiná $k$ por vía dinámica con tus datos y compará con el $k$ estático del Colab 05usando `compatibilidad()`. **Ésta comparación es la práctica de la clase**: dos métodosindependientes para la misma constante física.**9.3.** Si no son compatibles, revisá en este orden: (a) ¿incluiste la corrección por masa delresorte?; (b) ¿el resorte es el mismo?; (c) ¿la amplitud de oscilación está dentro del régimenlineal?; (d) ¿subestimaste alguna incerteza?**9.4.** Repetí la determinación del período usando los **mínimos** en lugar de los máximos(`find_peaks(-x, ...)`). Deberían dar lo mismo. Si no dan, tenés una asimetría en la señal que valela pena investigar.**9.5.** Adquirí la misma oscilación a 50 Hz y a 500 Hz. ¿Cuánto cambia $\\sigma_T$? ¿Vale la pena elarchivo diez veces más grande?